# Segmentació de caràcters de matrícules per projecció vertical.

Entrada : imatges {test}_box{n}.png de data/processed/
Sortida : fitxers {test}_box{n}_char{i}.png a data/chars/
          (32x64 px, fons negre, caràcter blanc, llests per a OCR)

In [ ]:
import cv2
import numpy as np
import argparse
import re
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.signal import find_peaks as _scipy_find_peaks


In [ ]:
# ─── Mides de sortida per a cada caràcter ────────────────────────────────────
CHAR_W = 32   # amplada canònica (píxels) del caràcter normalitzat
CHAR_H = 64   # alçada canònica

# ─── Alineació ───────────────────────────────────────────────────────────────
ALIGN_ANGLE_MAX = 15.0   # graus: si |angle| > aquest valor no rotem

# ─── Nombre de caràcters dinàmic ─────────────────────────────────────────────
N_CHARS_MIN = 5   # mínim de caràcters acceptats per matrícula vàlida
N_CHARS_MAX = 8   # màxim de caràcters acceptats per matrícula vàlida

# ─── Detecció de pics (detect_char_peaks) ────────────────────────────────────
PEAK_MIN_DISTANCE_RATIO  = 0.08   # separació mínima entre pics com a fracció de l'amplada total
PEAK_MIN_PROMINENCE_RATIO = 0.20  # prominència mínima com a fracció del màxim de la projecció

# ─── Filtratge de segments ────────────────────────────────────────────────────
SEG_MIN_WIDTH_ABS   = 8     # amplada mínima absoluta (px en coords ampliades)
SEG_MAX_WIDTH_RATIO = 0.25  # amplada màxima com a fracció de l'amplada total
SEG_MIN_DENSITY     = 0.05  # densitat mínima de píxels blancs sobre l'àrea del segment

# ─── Binarització adaptativa ─────────────────────────────────────────────────
ADAPTIVE_C = 5   # constant C de cv2.adaptiveThreshold (restada de la mitjana local de la finestra)

PROCESSED_DIR = "data/processed"
OUT_DIR       = "data/segmented"

## BLOC 1 — Preprocessament

In [ ]:
def preprocess(crop_bgr: np.ndarray) -> np.ndarray:
    """
    Prepara el crop per a la projecció:
      1. Escala de grisos
      2. medianBlur  → elimina soroll impulsiu (salt-i-pebre)
      3. Normalització → estira el contrast fins a [0, 255]
      4. Resize ×2.5  → fa els caràcters prou grans per segmentar bé
      5. GaussianBlur → suavitza artefactes del resize

    Retorna: imatge grisa uint8 ampliada.
    """
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 3)

    mn, mx = gray.min(), gray.max()
    if mx > mn:
        gray = ((gray - mn) / (mx - mn) * 255).astype(np.uint8)

    gray = cv2.resize(gray, None, fx=2.5, fy=2.5,
                      interpolation=cv2.INTER_CUBIC)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    return gray

def adaptive_binarize(gray: np.ndarray) -> np.ndarray:
    """
    Binaritza amb threshold adaptatiu gaussià i assegura que els caràcters
    siguin BLANCS.

    A diferència d'Otsu (llindar global únic), el threshold adaptatiu calcula
    un llindar diferent per a cada píxel en funció del seu entorn local.
    Això millora la robustesa davant de gradients d'il·luminació no uniformes
    (p.ex. plaques parcialment a l'ombra o amb reflexos localitzats).

    blockSize = max(11, (h // 4) | 1):
      - Proporcional a l'alçada de la imatge: la finestra de context creix
        amb la resolució de la placa (ja ampliada ×2.5 per preprocess).
      - L'operació '| 1' força que sigui senar (requeriment de OpenCV).
      - Mínim 11 px per evitar finestres massa petites amb massa soroll.

    THRESH_BINARY_INV: assumim caràcters foscos sobre fons clar.
      El llindar invertit retorna BLANC on hi ha tinta fosca.
      Si la imatge és al revés (caràcters clars), la regla final ho corregeix.

    ADAPTIVE_C = 5: constant que es resta del valor mig calculat per la
      finestra. Valors positius de C afavoreixen que els píxels lleugerament
      per sota de la mitjana es considerin fons i no text.
    """
    h = gray.shape[0]
    # blockSize ha de ser senar i ≥ 3; el '| 1' garanteix imparitat
    block_size = max(11, (h // 4) | 1)

    binary = cv2.adaptiveThreshold(
        gray,
        maxValue=255,
        adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        thresholdType=cv2.THRESH_BINARY_INV,
        blockSize=block_size,
        C=ADAPTIVE_C,
    )

    # Garantim que els caràcters siguin BLANCS (convenció de la projecció)
    if np.sum(binary == 255) < np.sum(binary == 0):
        binary = cv2.bitwise_not(binary)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    return binary

def binarize_otsu(gray: np.ndarray) -> np.ndarray:
    """
    Binaritza amb Otsu global i assegura que els caràcters siguin BLANCS.

    Otsu tria automàticament el llindar T* que maximitza la variància
    entre les dues classes (fons / text):

        T* = argmax_T  σ²_between(T)

    Conservada com a alternativa quan es prefereix un llindar únic global
    (p.ex. imatges amb il·luminació molt uniforme o per comparació).
    """
    _, binary = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if np.sum(binary == 255) < np.sum(binary == 0):
        binary = cv2.bitwise_not(binary)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    return binary


def binarize(gray: np.ndarray) -> np.ndarray:
    """
    Binaritza per a la segmentació (BLOC 3) usant threshold adaptatiu.

    Delega a adaptive_binarize, que és més robusta que Otsu davant de
    variacions locals d'il·luminació. Per a binarització global Otsu,
    usa binarize_otsu.
    """
    return adaptive_binarize(gray)

## BLOC 2 — Correcció d'inclinació (deskew)

In [ ]:
def deskew_by_baseline(crop_bgr: np.ndarray) -> tuple[np.ndarray, float, int]:
    """
    Corregeix la inclinació de la placa alineant els centroides dels caràcters.

    En comptes de minAreaRect (que opera sobre TOTS els píxels blancs i
    inclou fons i vores, retornant sovint ≈ 0°), aquest mètode:
      1. Detecta components connexes sobre la imatge binaritzada.
      2. Filtra les que no podrien ser caràcters (massa petites, massa grans,
         o tocant les vores de la placa).
      3. Ajusta una recta per regressió lineal als centroides (cx, cy).
      4. Obté l'angle del pendent m: α = arctan(m) · (180/π).

    Convenció de signes (coordenades imatge, y cap avall):
      - m > 0: la línia de text va cap avall-dreta → inclinació horària.
        Correccció: rotar en sentit antihorari → angle positiu en OpenCV.
      - m < 0: la línia de text va cap amunt-dreta → inclinació antihoraria.
        Correccció: rotar en sentit horari → angle negatiu en OpenCV.
    Per tant, angle_corr = arctan(m) s'aplica directament a getRotationMatrix2D.

    Filtratge de components:
      - Alçada: 30%–90% de l'alçada de la placa
      - Amplada: 3%–20% de l'amplada de la placa
      - No tocar les vores: marge mínim de 2 px a cada costat

    Si queden menys de 3 components, no es rota (senyal insuficient per
    estimar l'angle amb prou fiabilitat estadística).

    Retorna:
      aligned  : imatge BGR alineada (o l'original si no es rota)
      angle    : angle de correcció aplicat en graus (0.0 si no s'ha rotat)
      n_comps  : nombre de components usades per l'estimació (0 si no es rota)
    """
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    # Binaritzem amb Otsu per a la detecció de components (la imatge original,
    # no ampliada, no ha passat per preprocess, per tant Otsu és suficient aquí)
    _, binary = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.sum(binary == 255) < np.sum(binary == 0):
        binary = cv2.bitwise_not(binary)

    n_labels, _, stats, centroids = cv2.connectedComponentsWithStats(
        binary, connectivity=8)

    valid_cx: list[float] = []
    valid_cy: list[float] = []

    for label in range(1, n_labels):  # 0 és el fons
        x, y, w, h, _ = stats[label, :5]
        cx, cy = centroids[label]

        # Filtre per alçada: el caràcter ha d'ocupar entre el 30% i el 90%
        # de l'alçada de la placa (elimina punts de soroll i la placa sencera)
        if not (0.30 * H <= h <= 0.90 * H):
            continue

        # Filtre per amplada: entre el 3% i el 20% de l'amplada de la placa
        # (elimina components molt estretes —soroll— i molt amples —vores—)
        if not (0.03 * W <= w <= 0.20 * W):
            continue

        # Rebuig de components que toquen les vores (marge mínim 2 px)
        if x < 2 or y < 2 or (x + w) > (W - 2) or (y + h) > (H - 2):
            continue

        valid_cx.append(cx)
        valid_cy.append(cy)

    n_comps = len(valid_cx)
    if n_comps < 3:
        # Massa pocs caràcters detectats: no rotem per evitar errors d'estimació
        return crop_bgr, 0.0, 0

    # Regressió lineal als centroides: cy = m·cx + b
    # El pendent m dóna l'angle de la línia de text respecte a l'horitzontal
    m, _ = np.polyfit(valid_cx, valid_cy, 1)
    angle_corr = float(np.degrees(np.arctan(m)))

    if abs(angle_corr) > ALIGN_ANGLE_MAX:
        # Angle massa gran: probable error d'estimació, no rotem
        return crop_bgr, 0.0, n_comps

    H_img, W_img = crop_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((W_img / 2.0, H_img / 2.0), angle_corr, 1.0)
    aligned = cv2.warpAffine(crop_bgr, M, (W_img, H_img),
                             flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return aligned, angle_corr, n_comps

def deskew(crop_bgr: np.ndarray) -> tuple[np.ndarray, float]:
    """
    Àlies de deskew_by_baseline per compatibilitat amb codi existent.

    Retorna (aligned, angle_corr) sense el nombre de components usades.
    Per accedir a n_comps, crida deskew_by_baseline directament.
    """
    aligned, angle_corr, _ = deskew_by_baseline(crop_bgr)
    return aligned, angle_corr

## BLOC 3 — Segmentació per projecció vertical

In [ ]:
def vertical_projection(binary: np.ndarray) -> np.ndarray:
    """
    Calcula l'histograma de projecció vertical:
      proj[x] = nombre de píxels BLANCS a la columna x

    On hi ha caràcter → valor ALT.
    On hi ha espai entre caràcters → valor BAIX (valley).
    """
    return np.sum(binary == 255, axis=0).astype(np.float32)


def smooth_projection(proj: np.ndarray, k: int = 9) -> np.ndarray:
    """
    Mitjana mòbil de finestra k sobre la projecció.

    Sense suavitzat, el soroll residual genera micro-valleys espuris
    que confonen la cerca de fronteres. La finestra k=9 és un bon
    compromís: elimina variacions d'1-2 px però preserva les depressions
    reals entre caràcters (que solen tenir 5-20 px d'amplada).
    """
    kernel = np.ones((1, k), dtype=np.float32) / k
    return cv2.filter2D(proj.reshape(1, -1), -1, kernel).flatten()


def find_valley(proj: np.ndarray, left: int, right: int) -> int:
    """
    Retorna l'índex del mínim de proj en la finestra [left, right].

    La lògica és: cada frontera entre caràcters hauria d'estar a prop
    del punt equidistant entre dos caràcters consecutius. Buscar el
    mínim local en una finestra al voltant d'aquest punt garanteix
    que la tall cau on hi ha menys tinta.
    """
    segment = proj[left: right + 1]
    return left + int(np.argmin(segment))

def detect_char_peaks(proj_sm: np.ndarray) -> tuple[np.ndarray, list[int]]:
    """
    Detecta pics a la projecció vertical suavitzada i deriva les fronteres.

    Cada pic és el centre d'un caràcter candidat. Les fronteres es calculen
    com els valleys (mínim local de proj_sm) entre pics consecutius.

    Paràmetres de detecció:
      distance   = PEAK_MIN_DISTANCE_RATIO × W
                   Separació mínima entre dos pics (evita detectar el mateix
                   caràcter dues vegades per soroll intern).
      prominence = PEAK_MIN_PROMINENCE_RATIO × max(proj_sm)
                   Alçada mínima del pic respecte als seus valleys veïns
                   (filtra pics espuris en zones de soroll baix).

    Si scipy és disponible, s'usa scipy.signal.find_peaks que implementa
    l'algorisme de manera eficient. En cas contrari, s'usa una detecció
    manual equivalent: un pic és un índex i tal que proj[i] és el màxim
    dins d'una finestra ±distance i la seva prominència és suficient.

    Retorna:
      peaks      : array d'índexs dels pics detectats (centres de caràcters)
      boundaries : llista de fronteres [0, v₁, v₂, ..., W], on vᵢ és el
                   valley entre el pic i i el pic i+1
    """
    W = len(proj_sm)
    distance = max(1, int(W * PEAK_MIN_DISTANCE_RATIO))
    prominence = float(proj_sm.max()) * PEAK_MIN_PROMINENCE_RATIO

    peaks, _ = _scipy_find_peaks(
        proj_sm, distance=distance, prominence=prominence)

    if len(peaks) == 0:
        return np.array([], dtype=int), [0, W]

    # Fronteres com a valleys entre pics consecutius
    boundaries: list[int] = [0]
    for i in range(len(peaks) - 1):
        valley = find_valley(proj_sm, int(peaks[i]), int(peaks[i + 1]))
        boundaries.append(valley)
    boundaries.append(W)

    return peaks, boundaries

def filter_segments(segments: list, binary: np.ndarray) -> list:
    """
    Filtra segments que clarament no corresponen a caràcters vàlids.

    Criteris d'eliminació:
      - Amplada < SEG_MIN_WIDTH_ABS (8 px):
          segment massa estret, probablement és un artefacte o pic espuri.
      - Amplada > SEG_MAX_WIDTH_RATIO × W_total (25%):
          segment massa ample, probablement engloba dos caràcters enganxats
          o és una zona de fons captada per error.
      - Densitat de píxels blancs < SEG_MIN_DENSITY (5%) de l'àrea:
          la ROI té molt poca tinta, és quasi buit → no hi ha caràcter real.

    Retorna la llista filtrada de (x1, x2).
    """
    if not segments:
        return []

    total_w = binary.shape[1]
    filtered: list[tuple[int, int]] = []

    for x1, x2 in segments:
        w = x2 - x1

        if w < SEG_MIN_WIDTH_ABS:
            continue

        if w > total_w * SEG_MAX_WIDTH_RATIO:
            continue

        roi = binary[:, x1:x2]
        density = np.sum(roi == 255) / max(roi.size, 1)
        if density < SEG_MIN_DENSITY:
            continue

        filtered.append((x1, x2))

    return filtered

def segment_characters(gray: np.ndarray) -> tuple[list, np.ndarray, np.ndarray, np.ndarray]:
    """
    Segmenta la imatge grisa en regions de caràcters de forma dinàmica.

    El nombre de caràcters no és fix: es descobreix a partir dels pics de
    la projecció vertical suavitzada. Segments que no superen el filtre de
    qualitat (amplada, densitat) s'eliminen.

    Retorna:
      segments : llista de (x1, x2) filtrada per qualitat
      binary   : imatge binaritzada (per visualització i extracció)
      proj_sm  : projecció suavitzada (per visualització)
      peaks    : índexs dels pics detectats (per visualització)
    """
    binary  = binarize(gray)
    proj    = vertical_projection(binary)
    proj_sm = smooth_projection(proj, k=9)

    peaks, boundaries = detect_char_peaks(proj_sm)

    segments_raw = [
        (boundaries[i], boundaries[i + 1])
        for i in range(len(boundaries) - 1)
        if boundaries[i + 1] - boundaries[i] > 2
    ]

    segments = filter_segments(segments_raw, binary)
    return segments, binary, proj_sm, peaks


## BLOC 4 — Extracció i normalització de cada caràcter

In [ ]:
def extract_char_roi(binary: np.ndarray, x1: int, x2: int) -> np.ndarray:
    """
    Retalla la ROI d'un caràcter i ajusta el bounding box intern
    als píxels blancs (elimina marges buits).

    El padding de 1 px als quatre costats evita que el caràcter
    toqui el marge (ajuda el template matching posterior).
    """
    roi = binary[:, x1:x2]
    ys, xs = np.where(roi == 255)

    if len(xs) == 0:
        # Segment buit (possible fals segment entre caràcters)
        return roi

    pad = 1
    y1b = max(0, ys.min() - pad)
    y2b = min(roi.shape[0] - 1, ys.max() + pad)
    x1b = max(0, xs.min() - pad)
    x2b = min(roi.shape[1] - 1, xs.max() + pad)

    return roi[y1b: y2b + 1, x1b: x2b + 1]


def normalize_char(char_img: np.ndarray,
                   out_w: int = CHAR_W,
                   out_h: int = CHAR_H) -> np.ndarray:
    """
    Redimensiona el caràcter per encabir-lo en un llenç out_h x out_w,
    mantenint la proporció i centrant-lo.

    La mida fixa (32×64) és la que espera el template matching:
    totes les plantilles de la base de referència tenen aquesta mida,
    de manera que la comparació és sempre entre matrius del mateix rang.
    """
    if char_img is None or char_img.size == 0:
        return np.zeros((out_h, out_w), dtype=np.uint8)

    h, w = char_img.shape[:2]
    scale = min(out_w / max(w, 1), out_h / max(h, 1))
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))

    resized = cv2.resize(char_img, (new_w, new_h),
                         interpolation=cv2.INTER_AREA)
    canvas = np.zeros((out_h, out_w), dtype=np.uint8)
    x0 = (out_w - new_w) // 2
    y0 = (out_h - new_h) // 2
    canvas[y0: y0 + new_h, x0: x0 + new_w] = resized
    return canvas

## BLOC 5 — Pipeline principal

In [ ]:
def segment_plate(crop_bgr: np.ndarray) -> tuple[list[dict], np.ndarray, float]:
    """
    Pipeline complet per a un crop de matrícula.

    Ordre d'operacions:
      1. deskew_by_baseline → corregeix la inclinació per regressió sobre centroides
      2. preprocess         → grisos, blur, normalització, resize ×2.5
      3. segment_characters → projecció vertical + detect_char_peaks + filter_segments
      4. extract + normalize → retalla i redimensiona cada caràcter

    Validació del resultat:
      Si el nombre de segments finals no és dins [N_CHARS_MIN, N_CHARS_MAX]
      (és a dir, [5, 8]), el crop es considera rebutjat i es retorna una
      llista buida de caràcters. El caller pot detectar-ho amb len(chars) == 0.

    Retorna:
      chars      : llista de dicts, un per caràcter detectat:
                     {
                       'idx'      : int,          índex 0-based
                       'char_img' : np.ndarray,   ROI binaritzada ajustada
                       'norm_img' : np.ndarray,   32×64 px llest per OCR
                       'x1'       : int,          frontera esquerra (coords ampliades)
                       'x2'       : int,          frontera dreta    (coords ampliades)
                     }
                   Llista buida si el crop és rebutjat.
      aligned    : imatge BGR alineada (útil per visualització)
      angle_corr : angle de correcció aplicat en graus (0.0 si no s'ha rotat)
    """
    # Pas 1 — Correcció d'inclinació per alineació de centroides de caràcters
    aligned, angle_corr, _ = deskew_by_baseline(crop_bgr)

    # Pas 2 — Preprocessament estàndard
    gray = preprocess(aligned)

    # Pas 3 — Segmentació per projecció amb nombre dinàmic de caràcters
    segments, binary, proj, peaks = segment_characters(gray)

    # Validació: rebutgem si el nombre de segments és fora del rang [5, 8]
    n = len(segments)
    if not (N_CHARS_MIN <= n <= N_CHARS_MAX):
        return [], aligned, angle_corr

    # Pas 4 — Extracció i normalització de cada caràcter
    chars = []
    for i, (x1, x2) in enumerate(segments):
        roi  = extract_char_roi(binary, x1, x2)
        norm = normalize_char(roi)
        chars.append({
            'idx':      i,
            'char_img': roi,
            'norm_img': norm,
            'x1':       x1,
            'x2':       x2,
        })
    return chars, aligned, angle_corr


def parse_crop_filename(path: Path) -> tuple[str | None, int | None]:
    """Extreu (stem, box_idx) de noms com 'test_001_box2.png'."""
    m = re.match(r'^(.+)_box(\d+)$', path.stem)
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

## BLOC 6 — Visualització opcional

In [ ]:
def visualize_segmentation(crop_bgr: np.ndarray,
                           aligned: np.ndarray,
                           angle_corr: float,
                           binary: np.ndarray,
                           proj: np.ndarray,
                           chars: list[dict],
                           title: str = '',
                           n_comps: int = 0,
                           peaks: np.ndarray | None = None,
                           accepted: bool = True) -> None:
    """
    Mostra en una figura les cinc vistes principals:
      1. Crop original (abans del deskew)
      2. Imatge alineada (després del deskew) amb l'angle i components usades
      3. Binari amb les fronteres de segmentació
      4. Projecció vertical suavitzada amb pics (○ verds) i fronteres (-- roges)
      5. Caràcters normalitzats (32×64) en fila

    Paràmetres addicionals:
      n_comps  : nombre de components connexes usades per estimar l'angle de deskew
      peaks    : índexs dels pics de la projecció (per marcar-los al gràfic)
      accepted : si False, el títol es mostra amb fons taronja indicant rebuig
    """
    n = len(chars)
    fig = plt.figure(figsize=(max(16, n * 2 + 4), 7))

    # Títol principal amb indicació de rebuig si escau
    title_color = 'orangered' if not accepted else 'black'
    rejected_suffix = '  ✗ REBUTJAT (fora de rang [5–8])' if not accepted else ''
    fig.suptitle(title + rejected_suffix, fontsize=13, fontweight='bold',
                 color=title_color,
                 bbox=dict(facecolor='orange', alpha=0.25, pad=4) if not accepted else {})

    n_cols = max(n, 1) + 3  # original + aligned + binary + proj + n caràcters

    # ── Vista 1: crop original ────────────────────────────────────────────────
    ax1 = plt.subplot(2, n_cols, 1)
    ax1.imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    ax1.set_title('Original', fontsize=9)
    ax1.axis('off')

    # ── Vista 2: imatge alineada (post-deskew) ────────────────────────────────
    ax2 = plt.subplot(2, n_cols, 2)
    ax2.imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
    if angle_corr != 0.0:
        comps_label = f', {n_comps} comps' if n_comps > 0 else ''
        angle_label = f'Deskew ({angle_corr:+.1f}°{comps_label})'
    else:
        angle_label = 'Deskew (0° — ok)'
    ax2.set_title(angle_label, fontsize=9)
    ax2.axis('off')

    # ── Vista 3: binari + fronteres ───────────────────────────────────────────
    ax3 = plt.subplot(2, n_cols, 3)
    ax3.imshow(binary, cmap='gray')
    for c in chars:
        ax3.axvline(c['x1'], color='lime', linewidth=1, alpha=0.8)
        ax3.axvline(c['x2'], color='lime', linewidth=1, alpha=0.8)
    ax3.set_title('Binari + fronteres', fontsize=9)
    ax3.axis('off')

    # ── Vista 4: projecció vertical ───────────────────────────────────────────
    ax4 = plt.subplot(2, n_cols, n_cols + 1)
    ax4.plot(proj, color='steelblue', linewidth=1)
    ax4.fill_between(range(len(proj)), proj, alpha=0.25, color='steelblue')

    # Fronteres (valleys derivats) com a línies vermelles discontínues
    for c in chars:
        ax4.axvline(c['x1'], color='red', linewidth=1, linestyle='--', alpha=0.7)
    # Última frontera (x2 del darrer caràcter)
    if chars:
        ax4.axvline(chars[-1]['x2'], color='red', linewidth=1,
                    linestyle='--', alpha=0.7)

    # Pics detectats com a cercles verds
    if peaks is not None and len(peaks) > 0:
        ax4.plot(peaks, proj[peaks], 'o', color='limegreen',
                 markersize=6, label='pics', zorder=5)
        ax4.legend(fontsize=7, loc='upper right')

    ax4.set_title('Projecció vertical (suavitzada)', fontsize=9)
    ax4.set_xlabel('Columna x', fontsize=8)
    ax4.set_ylabel('Píxels blancs', fontsize=8)

    # ── Vista 5: caràcters normalitzats ───────────────────────────────────────
    for c in chars:
        ax = plt.subplot(2, n_cols, n_cols + 2 + c['idx'])
        ax.imshow(c['norm_img'], cmap='gray', vmin=0, vmax=255)
        ax.set_title(f"char {c['idx']}", fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

## Main

In [ ]:
processed_dir = Path(PROCESSED_DIR)
out_dir       = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

crop_files = sorted(processed_dir.glob('*_box*.png'))[:20]
print(f'Crops trobats : {len(crop_files)}')
print(f'Sortida       : {out_dir.resolve()}')
print(f'Rang caràcters: [{N_CHARS_MIN}, {N_CHARS_MAX}]')
print(f'Angle màx     : ±{ALIGN_ANGLE_MAX}°')
print('-' * 50)

n_ok = 0
n_skip = 0
n_reject = 0

for crop_path in crop_files:
        stem, box_idx = parse_crop_filename(crop_path)
        if stem is None:
            print(f'  SKIP (nom inesperat): {crop_path.name}')
            n_skip += 1
            continue

        crop_bgr = cv2.imread(str(crop_path))
        if crop_bgr is None:
            print(f'  ERROR llegint: {crop_path.name}')
            n_skip += 1
            continue

        # ── Deskew + Segmentació ─────────────────────────────────────────────
        chars, aligned, angle_corr = segment_plate(crop_bgr)

        accepted = len(chars) > 0

        if not accepted:
            angle_str = f'{angle_corr:+.1f}°' if angle_corr != 0.0 else '  0.0°'
            print(f'  REBUTJAT {crop_path.name:30s} angle={angle_str:>8}  '
                  f'→ fora del rang [{N_CHARS_MIN},{N_CHARS_MAX}]')
            n_reject += 1
        else:
            # ── Guardar caràcters normalitzats ───────────────────────────────
            for c in chars:
                out_name = f'{stem}_box{box_idx}_char{c["idx"]}.png'
                out_path = out_dir / out_name
                cv2.imwrite(str(out_path), c['norm_img'])

            angle_str = f'{angle_corr:+.1f}°' if angle_corr != 0.0 else '  0.0° (no rotat)'
            print(f'  OK  {crop_path.name:35s} angle={angle_str:>8}  → {len(chars)} chars')
            n_ok += 1

        # ── Visualització opcional ───────────────────────────────────────────
        # Re-executem les passes internes per obtenir les dades de visualització
        aligned_v, angle_v, n_comps_v = deskew_by_baseline(crop_bgr)
        gray_v = preprocess(aligned_v)
        segs_v, binary_v, proj_v, peaks_v = segment_characters(gray_v)

        # Reconstituïm chars per a visualització fins i tot en cas de rebuig
        chars_v: list[dict] = []
        for i, (x1, x2) in enumerate(segs_v):
            roi  = extract_char_roi(binary_v, x1, x2)
            norm = normalize_char(roi)
            chars_v.append({
                'idx': i, 'char_img': roi, 'norm_img': norm,
                'x1': x1, 'x2': x2,
            })

        visualize_segmentation(
            crop_bgr, aligned_v, angle_v,
            binary_v, proj_v, chars_v,
            title=crop_path.name,
            n_comps=n_comps_v,
            peaks=peaks_v,
            accepted=accepted,
        )

print('-' * 50)
print(f'Processats : {n_ok}  |  Rebutjats : {n_reject}  |  Saltats : {n_skip}')
print(f'Caràcters a: {out_dir.resolve()}')
print(f'Convenció  : {{stem}}_box{{n}}_char{{i}}.png  (32×{CHAR_H} px)')
